In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Stap 2: Tabblad inlezen als DataFrame
df = pd.read_excel('AmesHousing.xlsx', sheet_name='AmesHousing')

# Ontbrekende data verwijderen (voorzorgsmaatregel voor het model)
df = df.dropna()

# Bekijk de eerste 5 rijen
df.head()

,ID,SalePrice,Garage,Overall Qual,Gr Liv Area,Total Bsmt SF,Lot Area,Year Built,Full Bath,Bedroom AbvGr,Neighborhood,House Style
0,1,215000,yes,6,1656,1080.0,31770,1960,1,3,NAmes,1Story
1,2,105000,yes,5,896,882.0,11622,1961,1,2,NAmes,1Story
2,3,172000,yes,6,1329,1329.0,14267,1958,1,3,NAmes,1Story
3,4,244000,yes,7,2110,2110.0,11160,1968,2,3,NAmes,1Story
4,5,189900,yes,5,1629,928.0,13830,1997,2,3,Gilbert,2Story


In [6]:
# Stap 4: Data prepareren
# Target en onze top 3 features
X = df[['Overall Qual', 'Gr Liv Area', 'Neighborhood']].copy()
y = df['SalePrice']

# 4a: One-hot encoding voor categorische data (Neighborhood)
X_encoded = pd.get_dummies(X, columns=['Neighborhood'], drop_first=True)

# 4b: Data opsplitsen in 4 stukken (Train-test split)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Data schalen (belangrijk voor SGDRegressor)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
# Stap 5: Initialiseer SGDRegressor met hyperparameters
model_init = SGDRegressor(max_iter=1000, learning_rate='invscaling', eta0=0.01, random_state=42)

# Model trainen
model_init.fit(X_train_scaled, y_train)

# Stap 6: Model evalueren
y_pred = model_init.predict(X_test_scaled)

rmse_init = np.sqrt(mean_squared_error(y_test, y_pred))
r2_init = r2_score(y_test, y_pred)

print("--- Resultaten Initiële Run ---")
print(f"RMSE: {rmse_init:.2f}")
print(f"R-squared: {r2_init:.4f}")

--- Resultaten Initiële Run ---
RMSE: 42578.28
R-squared: 0.7567


In [8]:
# Stap 7: 
# Experiment 1: Meer features, meer epochs, lagere learning rate

# Nieuwe features toevoegen 
X_exp = df[['Overall Qual', 'Gr Liv Area', 'Neighborhood', 'Total Bsmt SF', 'Garage', 'House Style']].copy()

# One-hot encoding voor nieuwe categorische features
X_exp_encoded = pd.get_dummies(X_exp, columns=['Neighborhood', 'Garage', 'House Style'], drop_first=True)

# Opnieuw splitsen en schalen
X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(X_exp_encoded, y, test_size=0.2, random_state=42)

scaler_exp = StandardScaler()
X_train_exp_scaled = scaler_exp.fit_transform(X_train_exp)
X_test_exp_scaled = scaler_exp.transform(X_test_exp)

# Nieuwe hyperparameters instellen
model_exp = SGDRegressor(max_iter=5000, learning_rate='invscaling', eta0=0.001, random_state=42)
model_exp.fit(X_train_exp_scaled, y_train_exp)

# Nieuwe resultaten evalueren
y_pred_exp = model_exp.predict(X_test_exp_scaled)
rmse_exp = np.sqrt(mean_squared_error(y_test_exp, y_pred_exp))
r2_exp = r2_score(y_test_exp, y_pred_exp)

print("--- Resultaten Experiment 1 ---")
print(f"RMSE: {rmse_exp:.2f}")
print(f"R-squared: {r2_exp:.4f}")

--- Resultaten Experiment 1 ---
RMSE: 42646.60
R-squared: 0.7559


In [9]:
# --- EXPERIMENT 2 ---
# Alleen numerieke features gebruiken 
# Kijken of het model werkt zonder categorische data
X_exp2 = df[['Overall Qual', 'Gr Liv Area', 'Total Bsmt SF', 'Year Built']].copy()

X_train_exp2, X_test_exp2, y_train_exp2, y_test_exp2 = train_test_split(X_exp2, y, test_size=0.2, random_state=42)

scaler_exp2 = StandardScaler()
X_train_exp2_scaled = scaler_exp2.fit_transform(X_train_exp2)
X_test_exp2_scaled = scaler_exp2.transform(X_test_exp2)

# Andere learning rate methode ('constant') proberen
model_exp2 = SGDRegressor(max_iter=2000, learning_rate='constant', eta0=0.005, random_state=42)
model_exp2.fit(X_train_exp2_scaled, y_train_exp2)

y_pred_exp2 = model_exp2.predict(X_test_exp2_scaled)
rmse_exp2 = np.sqrt(mean_squared_error(y_test_exp2, y_pred_exp2))
r2_exp2 = r2_score(y_test_exp2, y_pred_exp2)

print("--- Resultaten Experiment 2 (Alleen Numeriek) ---")
print(f"RMSE: {rmse_exp2:.2f}")
print(f"R-squared: {r2_exp2:.4f}")

--- Resultaten Experiment 2 (Alleen Numeriek) ---
RMSE: 48187.95
R-squared: 0.6884


In [10]:
# --- EXPERIMENT 3 ---
# Neighborhood teruggezet, 'Year Built' en 'Full Bath' toegevoegd
X_exp3 = df[['Overall Qual', 'Gr Liv Area', 'Neighborhood', 'Year Built', 'Full Bath']].copy()
X_exp3_encoded = pd.get_dummies(X_exp3, columns=['Neighborhood'], drop_first=True)

X_train_exp3, X_test_exp3, y_train_exp3, y_test_exp3 = train_test_split(X_exp3_encoded, y, test_size=0.2, random_state=42)

scaler_exp3 = StandardScaler()
X_train_exp3_scaled = scaler_exp3.fit_transform(X_train_exp3)
X_test_exp3_scaled = scaler_exp3.transform(X_test_exp3)

# Lagere max_iter en standaard learning rate
model_exp3 = SGDRegressor(max_iter=500, learning_rate='invscaling', eta0=0.01, random_state=42)
model_exp3.fit(X_train_exp3_scaled, y_train_exp3)

y_pred_exp3 = model_exp3.predict(X_test_exp3_scaled)
rmse_exp3 = np.sqrt(mean_squared_error(y_test_exp3, y_pred_exp3))
r2_exp3 = r2_score(y_test_exp3, y_pred_exp3)

print("--- Resultaten Experiment 3 (Gemixte Features) ---")
print(f"RMSE: {rmse_exp3:.2f}")
print(f"R-squared: {r2_exp3:.4f}")

--- Resultaten Experiment 3 (Gemixte Features) ---
RMSE: 42791.79
R-squared: 0.7542
